# 3. Monte Carlo tournament

Monte Carlo simulation repeats the same experiment many times while sampling random events. We introduce implementation error: intended actions sometimes flip.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

payoffs = {('C','C'):(3,3), ('C','D'):(0,5), ('D','C'):(5,0), ('D','D'):(1,1)}
def noisy(action, error=0.05):
    return ('D' if action == 'C' else 'C') if np.random.random() < error else action

In [ ]:
def run(strategy_a, strategy_b, rounds=100, error=0.05):
    ha, hb, pa, pb = [], [], 0, 0
    for _ in range(rounds):
        a = noisy(strategy_a(ha,hb), error); b = noisy(strategy_b(hb,ha), error)
        x,y = payoffs[(a,b)]; ha.append(a); hb.append(b); pa += x; pb += y
    return pa, pb

def ac(own, other): return 'C'
def ad(own, other): return 'D'
def tft(own, other): return 'C' if len(other)==0 else other[-1]
def rand(own, other): return np.random.choice(['C','D'])
strategies = {'always cooperate':ac, 'always defect':ad, 'tit for tat':tft, 'random':rand}

records=[]
for rep in range(500):
    for name, strat in strategies.items():
        opponents = [v for k,v in strategies.items() if k != name]
        total = sum(run(strat, opp, error=0.05)[0] for opp in opponents)
        records.append({'replication':rep, 'strategy':name, 'payoff':total})
mc = pd.DataFrame(records)
mc.groupby('strategy').payoff.describe()

In [ ]:
sns.boxplot(data=mc, x='payoff', y='strategy')
plt.title('Monte Carlo tournament under action error');